# 01 — XAI Landscape

Explainable AI sits at the intersection of model performance and human understanding. The field organises along several axes.

## Taxonomy of Explanation Methods

**Local vs Global**
- *Local*: explains a single prediction — why did the model classify *this* sample as X?
- *Global*: explains the model's overall behaviour — what does the model rely on in general?

**Post-hoc vs Intrinsic**
- *Intrinsic*: model is interpretable by design — decision trees, linear models, GAMs
- *Post-hoc*: explanation added after training — SHAP, LIME, Grad-CAM applied to black-box models

**Model-Agnostic vs Model-Specific**
- *Model-agnostic*: LIME, SHAP (KernelSHAP) — treats model as a black box
- *Model-specific*: Grad-CAM (requires gradients), TreeSHAP (exploits tree structure)

**Faithfulness vs Interpretability**
An explanation can be easy to read but unfaithful — it describes a simpler proxy, not the actual model. LIME explicitly approximates; TreeSHAP is exact.

## Historical Context

Early feature importance methods — permutation importance, coefficient magnitude in linear models — were heuristics that ignored interaction effects and depended on scale. LIME (Ribeiro et al., 2016) was the first widely adopted general-purpose local explainer; it fitted a local linear surrogate in the neighbourhood of each prediction. Vanilla gradient saliency (magnitude of ∂L/∂x) suffered from gradient saturation and noise. SHAP unified many prior methods under the Shapley value framework (Lundberg & Lee, 2017), providing axiom-backed guarantees that earlier methods lacked.

In [ ]:
# XAI method selector decision tree
import numpy as np

def xai_selector(model_type, data_type, scope, need_exact):
    """Recommend an XAI method based on context."""
    recs = []

    if scope == 'local':
        if model_type == 'tree' and need_exact:
            recs.append(('TreeSHAP', 'Exact Shapley values, O(TLD) per sample'))
        if data_type == 'image':
            recs.append(('Grad-CAM', 'Visual heatmap; requires gradient access'))
            recs.append(('Integrated Gradients', 'Axiom-satisfying attribution; requires gradient access'))
        if model_type in ('any', 'neural_net'):
            recs.append(('KernelSHAP', 'Model-agnostic; approximate; slow on large inputs'))
            recs.append(('LIME', 'Fast local surrogate; less stable than SHAP'))
        if data_type == 'tabular':
            recs.append(('DiCE Counterfactuals', 'Actionable recourse; shows what to change'))
    else:  # global
        if model_type == 'tree':
            recs.append(('TreeSHAP summary plot', 'Global feature importance from Shapley'))
        recs.append(('PDP / ALE', 'Marginal effect of each feature; ALE handles correlation'))
        recs.append(('Global surrogate', 'Decision tree approximating the full model'))
        if data_type == 'image':
            recs.append(('TCAV', 'Test with concept activation vectors'))

    return recs

# Demo the selector
scenarios = [
    ('tree', 'tabular', 'local', True),
    ('neural_net', 'image', 'local', False),
    ('any', 'tabular', 'global', False),
]

for args in scenarios:
    model_type, data_type, scope, need_exact = args
    print(f'\nScenario: {model_type} | {data_type} | {scope} | exact={need_exact}')
    for name, desc in xai_selector(*args):
        print(f'  -> {name}: {desc}')

In [ ]:
# Faithfulness vs Stability visualisation
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
n_methods = 6
methods = ['LIME', 'KernelSHAP', 'TreeSHAP', 'IG', 'Grad-CAM', 'Saliency']
faithfulness = [0.65, 0.82, 0.99, 0.90, 0.75, 0.55]
stability = [0.55, 0.80, 0.99, 0.88, 0.70, 0.40]
speed = [0.85, 0.30, 0.95, 0.60, 0.90, 0.99]

fig, ax = plt.subplots(figsize=(8, 5))
scatter = ax.scatter(faithfulness, stability, s=[s*500 for s in speed],
                     c=range(n_methods), cmap='tab10', alpha=0.8)
for i, m in enumerate(methods):
    ax.annotate(m, (faithfulness[i]+0.01, stability[i]+0.01), fontsize=10)
ax.set_xlabel('Faithfulness (1=exact)')
ax.set_ylabel('Stability (1=deterministic)')
ax.set_title('XAI Method Comparison\n(bubble size = speed)')
ax.set_xlim(0.4, 1.1)
ax.set_ylim(0.3, 1.1)
plt.tight_layout()
plt.savefig('/tmp/xai_landscape.png', dpi=100, bbox_inches='tight')
plt.show()
print('XAI landscape chart saved')